# Deploy candidate: COUPLED (alpha=1) masked model

The masked ablation showed coupled (auxiliary gradients into the trunk) beats the routed
alpha=0 model on accuracy and auxiliary quality. This trains coupled as a **deploy candidate**:
5 seeds, 150 epochs, masked, fs=0-gated checkpoint selection. It **saves `coupled.pt`**,
reports per-seed false-safe (deploy is gated on 0-false-safe), and computes the full deployed
metric suite (`coupled_metrics.json`) so the download is everything needed to swap the paper.
Runtime -> GPU, Run all. Reads `smart_load_shield_boost` from Drive (no upload).

In [ ]:
from google.colab import drive
import glob, os, sys
drive.mount('/content/drive')
c = glob.glob('/content/drive/MyDrive/**/smart_load_shield_boost', recursive=True)
FOLDER = c[0] if c else '/content/drive/MyDrive/smart_load_shield_boost'
sys.path.insert(0, FOLDER)
need = ['contingency_data.npz', 'grouped_split.npz', 'boost_core.py']
missing = [f for f in need if not os.path.exists(os.path.join(FOLDER, f))]
assert not missing, 'missing in ' + FOLDER + ': ' + str(missing)
assert 'base_mask' in open(os.path.join(FOLDER, 'boost_core.py')).read(), \
    'boost_core.py in FOLDER is the OLD unmasked version -- replace with round-6 masked'
print('FOLDER =', FOLDER, '| masked boost_core OK')

In [ ]:
import numpy as np, torch, json
import boost_core as B
from sklearn.metrics import roc_auc_score, average_precision_score
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device', DEV)
d = B.build_data(os.path.join(FOLDER, 'contingency_data.npz'),
                 os.path.join(FOLDER, 'grouped_split.npz'), DEV)
SEEDS = [0, 1, 2, 3, 4]

# ---- train COUPLED (alpha=1) x5 seeds, keep best val-at-fs0 checkpoint ----
best_val, best_state, best_seed = -1.0, None, None
per_seed = []
for sd in SEEDS:
    te, model = B.train_eval(d, seed=sd, alpha=1.0, branched=False, stack=False, vbus=False,
                             epochs=150, amp=(DEV == 'cuda'))
    val = B.evaluate(model, d, d['idx_va'])
    per_seed.append(dict(seed=sd, test_acc=te['acc'], test_fs=te['false_safe'],
                         val_acc=val['acc'], val_fs=val['false_safe'], vmin_mae=te['vmin_mae']))
    print('  coupled seed%d  test acc %.2f%%  test fs %.3f%%  val acc %.2f%%  val fs %.3f%%'
          % (sd, te['acc']*100, te['false_safe']*100, val['acc']*100, val['false_safe']*100))
    if val['false_safe'] == 0.0 and val['acc'] > best_val:
        best_val = val['acc']; best_state = {k: v.clone() for k, v in model.state_dict().items()}; best_seed = sd
assert best_state is not None, 'no coupled seed reached 0 false-safe on val -- do NOT deploy coupled'
torch.save(best_state, os.path.join(FOLDER, 'coupled.pt'))
print('\nsaved coupled.pt (best val-at-fs0 = seed %d, val acc %.2f%%)' % (best_seed, best_val*100))
print('per-seed test false-safe:', [round(p['test_fs']*100, 3) for p in per_seed], '%  (deploy needs the picked seed at 0)')

In [ ]:
# ---- full deployed metrics on the selected coupled checkpoint ----
m = B.CSGNNv2(node_dim=4, edge_dim=3, hidden=128, heads=4, branched=False, stack=False, vbus=False).to(DEV)
m.load_state_dict(best_state); m.set_base_mask(d['BASE']); m.eval()
idx = d['idx_te']
P, prob, VM, VU = [], [], [], []
with torch.no_grad():
    for s in range(0, len(idx), 4096):
        b = idx[s:s+4096]; r, vm, vu, dv, vb = m(d['NF'][b], d['DENSE'][b], d['ADJ'][b])
        P.append(r.argmax(1)); prob.append(torch.softmax(r, 1)); VM.append(vm); VU.append(torch.sigmoid(vu))
P = torch.cat(P); prob = torch.cat(prob).cpu().numpy(); VM = torch.cat(VM); VU = torch.cat(VU); R = d['Tr'][idx]
cm = torch.zeros(3, 3, dtype=torch.long)
for t, p in zip(R.cpu(), P.cpu()): cm[t, p] += 1
cm = cm.numpy()
acc = float((P == R).float().mean()); recall = [float(cm[c, c]/max(1, cm[c].sum())) for c in range(3)]
false_safe = float(cm[2, 0]/max(1, cm[2].sum()))
f1s = []
for k in range(3):
    tp = cm[k, k]; fp = cm[:, k].sum()-tp; fn = cm[k, :].sum()-tp
    pr = tp/max(1, tp+fp); rc = tp/max(1, tp+fn); f1s.append(2*pr*rc/max(1e-9, pr+rc))
macro_f1 = float(np.mean(f1s))
conv = (d['Td'][idx] == 0).cpu().numpy()
vt = d['Tv'][idx].cpu().numpy()[conv]; vp = VM.cpu().numpy()[conv]
vmae = float(np.abs(vt-vp).mean()); r2 = float(1-((vt-vp)**2).sum()/((vt-vt.mean())**2).sum())
r2fit = float(np.corrcoef(vt, vp)[0, 1]**2)
ut = d['Tu'][idx].cpu().numpy()[conv].ravel(); up = VU.cpu().numpy()[conv].ravel()
vuln_auc = float(roc_auc_score(ut, up)); vuln_prauc = float(average_precision_score(ut, up))
Y = R.cpu().numpy(); bins = np.linspace(0, 1, 16); cf = prob.max(1); cor = (prob.argmax(1) == Y)
ece = mce_g = 0.0
for i in range(15):
    msk = (cf > bins[i]) & (cf <= bins[i+1]); n = int(msk.sum())
    if n:
        gap = abs(cor[msk].mean()-cf[msk].mean()); ece += msk.mean()*gap
        if n >= 20: mce_g = max(mce_g, gap)
Yoh = np.eye(3)[Y]; brier = float(np.mean(((prob-Yoh)**2).sum(1))); nll = float(-np.mean(np.log(prob[np.arange(len(Y)), Y]+1e-12)))
roc = [float(roc_auc_score((Y == c).astype(int), prob[:, c])) for c in range(3)]
prc = [float(average_precision_score((Y == c).astype(int), prob[:, c])) for c in range(3)]
nonconv = B.divergence_report(m, d, idx)
out = dict(model='coupled', deploy_seed=best_seed, test_acc=acc, macro_f1=macro_f1, recall=recall,
           false_safe=false_safe, confusion=cm.tolist(), n_test=int(len(idx)), n_unstable=int(cm[2].sum()),
           umarg=int(cm[2, 1]), vmin_mae=vmae, vmin_r2=r2, vmin_r2_fit=r2fit, vuln_auc=vuln_auc,
           vuln_prauc=vuln_prauc, ece=float(ece), mce_guarded=float(mce_g), brier=brier, nll=nll,
           roc_auc=roc, pr_auc=prc, nonconv=nonconv, per_seed=per_seed)
json.dump(out, open(os.path.join(FOLDER, 'coupled_metrics.json'), 'w'), indent=2)
print('DEPLOYED coupled: acc %.2f%%  fs %.3f%% (%d/%d)  F1 %.2f%%  R2 %.3f  vulnAUC %.3f  ECE %.3f'
      % (acc*100, false_safe*100, cm[2, 0], cm[2].sum(), macro_f1*100, r2, vuln_auc, ece))
print('recall S/M/U %.1f/%.1f/%.1f  confusion %s' % (recall[0]*100, recall[1]*100, recall[2]*100, cm.tolist()))
print('nonconv F1 %.3f ROC-AUC %.4f  | saved coupled_metrics.json + coupled.pt to' % (nonconv['f1'], nonconv['roc_auc']), FOLDER)
if false_safe > 0:
    print('\n*** WARNING: deployed coupled checkpoint is NOT 0-false-safe (%d misses) -- do NOT deploy; keep routed. ***' % cm[2, 0])